# Module Info Scraping

This notebook combines the existing NUS, SMU, and SUTD module scrapers into one place and saves the outputs into the `DSA4264 Project` folder.

In [ ]:
import os
from pathlib import Path

# Get notebook directory and construct data path
NOTEBOOK_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in dir() else os.getcwd()
PROJECT_ROOT = Path(NOTEBOOK_DIR).parent.parent  # Go up to dsa4264/
OUTPUT_DIR = PROJECT_ROOT / "DSA4264 Data" / "NUS-SMU-SUTD Courses"

# Create output directory if it doesn't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

## NUS Modules

In [ ]:
from pathlib import Path
import json
import requests
import pandas as pd

BASE_URL = 'https://api.nusmods.com/v2'
TARGET_ACAD_YEAR = '2025-2026'
SESSION = requests.Session()
SESSION.headers.update({'User-Agent': 'Mozilla/5.0'})


def clean_text(value):
    if value is None:
        return None
    text = str(value).strip()
    return text if text else None


def download_module_info(acad_year):
    url = f'{BASE_URL}/{acad_year}/moduleInfo.json'
    response = SESSION.get(url, timeout=120)
    response.raise_for_status()
    return response.json()


def join_semesters(semester_data):
    if not semester_data:
        return None
    semesters = []
    for item in semester_data:
        semester = item.get('semester')
        if semester is not None:
            semesters.append(str(semester))
    return '; '.join(sorted(set(semesters), key=lambda x: int(x))) if semesters else None


def format_workload(workload):
    if not workload:
        return None
    return '; '.join(str(x) for x in workload)


def combine_requisites(row):
    parts = []
    if row.get('prerequisite'):
        parts.append(f"Prerequisite: {row['prerequisite']}")
    if row.get('corequisite'):
        parts.append(f"Corequisite: {row['corequisite']}")
    if row.get('preclusion'):
        parts.append(f"Preclusion: {row['preclusion']}")
    return ' | '.join(parts) if parts else None


acad_year = TARGET_ACAD_YEAR
raw_data = download_module_info(acad_year)

rows = []
for module in raw_data:
    row = {
        'acad_year': acad_year,
        'module_code': clean_text(module.get('moduleCode')),
        'module_name': clean_text(module.get('title')),
        'department': clean_text(module.get('department')),
        'faculty': clean_text(module.get('faculty')),
        'description': clean_text(module.get('description')),
        'module_credit': clean_text(module.get('moduleCredit')),
        'grading_basis': clean_text(module.get('gradingBasisDescription')),
        'workload': format_workload(module.get('workload')),
        'semesters_offered': join_semesters(module.get('semesterData')),
        'prerequisite': clean_text(module.get('prerequisite')),
        'corequisite': clean_text(module.get('corequisite')),
        'preclusion': clean_text(module.get('preclusion')),
        'prerequisite_rule': clean_text(module.get('prerequisiteRule')),
        'corequisite_rule': clean_text(module.get('corequisiteRule')),
        'preclusion_rule': clean_text(module.get('preclusionRule')),
        'additional_information': clean_text(module.get('additionalInformation')),
        'attributes': json.dumps(module.get('attributes'), ensure_ascii=False) if module.get('attributes') else None,
    }
    row['requisites'] = combine_requisites(row)
    rows.append(row)

nus_df = pd.DataFrame(rows)
preferred_order = [
    'acad_year', 'module_code', 'module_name', 'department', 'faculty', 'module_credit',
    'grading_basis', 'description', 'requisites', 'prerequisite', 'corequisite', 'preclusion',
    'semesters_offered', 'workload', 'attributes', 'additional_information',
    'prerequisite_rule', 'corequisite_rule', 'preclusion_rule'
]
nus_df = nus_df[preferred_order]
nus_output = OUTPUT_DIR / 'all_nus_modules.csv'
nus_df.to_csv(nus_output, index=False)
print(f'Academic year used: {acad_year}')
print(f'Saved {len(nus_df):,} rows to {nus_output}')
nus_df.head()


## SMU Modules

In [ ]:
from pathlib import Path
import pandas as pd
from playwright.async_api import async_playwright

TARGET_URL = 'https://courses.smu.edu.sg/courses?page=1&cq='
DOWNLOAD_DIR = OUTPUT_DIR / 'smu_catalog_downloads'
DOWNLOAD_DIR.mkdir(exist_ok=True)


def clean_text(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    return text if text else None


def clean_catalog_number(value):
    text = clean_text(value)
    if text is None:
        return None
    if text.endswith('.0'):
        text = text[:-2]
    return text


async def download_smu_catalog_csv():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(accept_downloads=True)
        page = await context.new_page()
        await page.goto(TARGET_URL, wait_until='domcontentloaded', timeout=120000)
        await page.wait_for_selector('button[data-test="export-all-results-as-csv"]', state='attached', timeout=120000)
        await page.wait_for_function(
            """
            () => {
                const headings = Array.from(document.querySelectorAll('h2'));
                const resultsHeading = headings.find(el => el.textContent.includes('Results ('));
                return resultsHeading && !resultsHeading.textContent.includes('Results (0)');
            }
            """,
            timeout=120000,
        )
        await page.evaluate(
            """
            () => {
                const btn = document.querySelector('button[data-test="export-all-results-as-csv"]');
                if (!btn) {
                    throw new Error('Export button not found');
                }
                btn.style.display = 'block';
                btn.style.visibility = 'visible';
                btn.style.opacity = '1';
            }
            """
        )
        button = page.locator('button[data-test="export-all-results-as-csv"]')
        async with page.expect_download(timeout=120000) as download_info:
            await button.evaluate('el => el.click()')
        download = await download_info.value
        filename = download.suggested_filename or 'smu_courses_raw.csv'
        raw_path = DOWNLOAD_DIR / filename
        await download.save_as(str(raw_path))
        await browser.close()
        return raw_path


raw_csv_path = await download_smu_catalog_csv()
raw_df = pd.read_csv(raw_csv_path)
raw_df.columns = [str(col).strip() for col in raw_df.columns]

smu_df = raw_df.copy()
if 'Catalog Number' in smu_df.columns:
    smu_df['Catalog Number'] = smu_df['Catalog Number'].map(clean_catalog_number)
if 'Subject Area' in smu_df.columns:
    smu_df['Subject Area'] = smu_df['Subject Area'].map(clean_text)
if {'Subject Area', 'Catalog Number'}.issubset(smu_df.columns):
    smu_df['module_code'] = (smu_df['Subject Area'].fillna('') + smu_df['Catalog Number'].fillna('')).str.strip()
    smu_df.loc[smu_df['module_code'] == '', 'module_code'] = None

rename_map = {
    'Course Title': 'module_name',
    'Offering Unit/Department': 'department',
    'Subject Area': 'subject_area',
    'Catalog Number': 'catalog_number',
    'Course Description': 'description',
    'Requisites': 'requisites',
    'Course (UG/PG)': 'course_level',
    'Grading Basis (e.g. Graded or Pass/Fail)': 'grading_basis',
    'Course Attributes': 'course_attributes',
    'Discipline Specific (Up to 5) - the dropdown list changes based on the Offering Unit/Department selection.': 'discipline_specific_competencies',
    'Standard Learning Outcomes (for SOA only)': 'standard_learning_outcomes',
    'SMU Graduate Learning Outcomes': 'graduate_learning_outcomes',
}

smu_df = smu_df.rename(columns=rename_map)
preferred_order = [
    'module_code', 'module_name', 'department', 'subject_area', 'catalog_number',
    'course_level', 'grading_basis', 'description', 'requisites', 'course_attributes',
    'discipline_specific_competencies', 'standard_learning_outcomes', 'graduate_learning_outcomes'
]
existing_preferred = [col for col in preferred_order if col in smu_df.columns]
remaining_cols = [col for col in smu_df.columns if col not in existing_preferred]
smu_df = smu_df[existing_preferred + remaining_cols]
smu_output = OUTPUT_DIR / 'all_smu_modules.csv'
smu_df.to_csv(smu_output, index=False)
print(f'Saved {len(smu_df):,} rows to {smu_output}')
smu_df.head()

## SUTD Modules

In [ ]:
import re
import time
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE = 'https://www.sutd.edu.sg'
LIST_URL = f'{BASE}/education/undergraduate/courses/'
session = requests.Session()
session.headers.update({'User-Agent': 'Mozilla/5.0'})


def clean(text):
    return re.sub(r'\s+', ' ', text).strip() if text else None


def get_soup(url):
    response = session.get(url, timeout=30)
    response.raise_for_status()
    return BeautifulSoup(response.text, 'html.parser')


def parse_list_page(page_num):
    url = LIST_URL if page_num == 1 else f'{LIST_URL}?paged={page_num}#general-listing'
    soup = get_soup(url)
    rows = []
    cards = soup.select('div.block.relative.bg-global-surface-secondary-light')

    for card in cards:
        link_el = card.find('a', href=re.compile(r'/course/'))
        title_el = card.select_one('div.text-h5')
        category_el = card.select_one('div.text-body2')
        pillar_el = card.select_one('#pillar')

        if not link_el or not title_el:
            continue

        title = clean(title_el.get_text(' ', strip=True))
        match = re.match(r'^(\d{2}\.\d{3}[A-Za-z-]*)\s+(.*)$', title)
        info_bits = [clean(x.get_text(' ', strip=True)) for x in card.select('div.text-body3')]
        instructors = [clean(a.get_text(' ', strip=True)) for a in card.select('span.richText a')]

        rows.append({
            'module_code': match.group(1) if match else None,
            'module_name': match.group(2) if match else title,
            'category': clean(category_el.get_text(' ', strip=True)) if category_el else None,
            'pillar': clean(pillar_el.get_text(' ', strip=True)) if pillar_el else None,
            'term': next((x.replace('Term(s)', '').strip() for x in info_bits if x.startswith('Term(s)')), None),
            'credits': next((x.replace('credits', '').strip() for x in info_bits if 'credits' in x.lower()), None),
            'instructors': '; '.join(filter(None, instructors)) or None,
            'detail_url': urljoin(BASE, link_el['href']),
        })

    return rows


def extract_section(main, heading):
    if main is None:
        return None
    for h5 in main.select('h5'):
        section_title = clean(h5.get_text(' ', strip=True))
        if section_title and section_title.lower() == heading.lower():
            parts = []
            node = h5.find_next_sibling()
            while node and getattr(node, 'name', None) != 'h5':
                text = clean(node.get_text(' ', strip=True))
                if text:
                    parts.append(text)
                node = node.find_next_sibling()
            return ' '.join(parts) if parts else None
    return None


def parse_detail(detail_url):
    soup = get_soup(detail_url)
    main = soup.select_one('main')
    description_parts = []

    if main is not None:
        for el in main.find_all(['p', 'h5']):
            text = clean(el.get_text(' ', strip=True))
            if not text:
                continue
            if el.name == 'h5' and text.lower() == 'prerequisite':
                break
            if el.name == 'p' and not text.startswith('Number of credits:'):
                description_parts.append(text)

    return {
        'description': ' '.join(description_parts) if description_parts else None,
        'prerequisite': extract_section(main, 'Prerequisite'),
        'tags': '; '.join(filter(None, (clean(a.get_text(' ', strip=True)) for a in soup.select('section#page-info a')))) or None,
    }


all_rows = []
seen_urls = set()
page = 1

while True:
    rows = parse_list_page(page)
    new_rows = [row for row in rows if row['detail_url'] not in seen_urls]
    if not new_rows:
        break

    for row in new_rows:
        seen_urls.add(row['detail_url'])
        try:
            row.update(parse_detail(row['detail_url']))
        except Exception as exc:
            row.update({'description': None, 'prerequisite': None, 'tags': None, 'detail_error': str(exc)})
        all_rows.append(row)
        time.sleep(0.5)

    print(f'Scraped page {page}, total modules so far: {len(all_rows)}')
    page += 1
    time.sleep(0.5)

sutd_df = pd.DataFrame(all_rows)
sutd_output = OUTPUT_DIR / 'all_sutd_modules.csv'
sutd_df.to_csv(sutd_output, index=False)
print(f'Saved {len(sutd_df):,} rows to {sutd_output}')
sutd_df.head()